# Lab 10 - Testowanie hipotez cz. 2

## Test chi^2

In [5]:
from scipy.stats import chisquare, chi2_contingency
import numpy as np


obs = np.array([20, 25, 15])  # wartość obserwowana
exp = np.array([1/3, 1/3, 1/3]) * np.sum(obs)  # oczekiwana liczba wystąpień (suma musi być taka jak dla obs!!!)
chi2, p = chisquare(obs, exp) 
print(f"Chi-squared: {chi2}, p-value: {p}")

Chi-squared: 2.5, p-value: 0.2865047968601901


### Relacja pomiędzy kolorem włosów a kolorem oczu

In [ ]:
eyes_vs_hair = [[438, 228, 115, 16], [1387, 746, 946, 53], [807, 189, 1768, 47]] # 3 wiersze odpowiadające kolorom oczu, 4 kolumny kolorom włosów
res = chi2_contingency(eyes_vs_hair)
print(f"Chi-squared: {res.statistic}, p-value: {res.pvalue}")

Chi-squared: 1010.0166269473259, p-value: 6.094851771482916e-215


## Test Anova 1-parametryczny

Procedura Anova: 1. Sprawdź hipotezę o normalności rozkładów, oraz sprawdź czy wariancje są zbliżone.

In [44]:
from scipy.stats import normaltest

data = [[1200, 100, 890], [1000, 1100, 650], [980, 700, 110], [900, 800, 900],
        [750, 500, 400], [800, 700, 350], [850, 750, 450], [500, 850, 500]]

res = normaltest(data)  
print(f"Statistic: {res.statistic}, p-value: {res.pvalue}")


Statistic: [0.99817627 3.95209211 0.10010418], p-value: [0.60708398 0.13861624 0.95117988]


2. Dla serii danych wykonaj test one-way Anova, sprawdzając hipotezę H0, że nie ma istotnych różnic:

In [15]:
from scipy.stats import f_oneway

data1, data2, data3 = data[0], data[1], data[2]

f_value, p_value = f_oneway(data1, data2, data3)
print(f'F-stat: {f_value}, p-val: {p_value}')

F-stat: 0.40456997042278037, p-val: 0.684190392702389




3. Jeśli prawdopodobieństwo testowe (`p_value`) pozwala odrzucić hipotezę zerową, przeprowadź test post-hoc Tukey’a umożliwiający określenie pomiędzy którymi parami różnice są istotne

In [16]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd


print(pairwise_tukeyhsd(np.concatenate([data1, data2, data3]), np.concatenate([['data1']*len(data1), ['data2']*len(data2), ['data3']*len(data3)])))

   Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2  meandiff p-adj    lower      upper   reject
----------------------------------------------------------
 data1  data2  186.6667 0.8636  -909.8967 1283.2301  False
 data1  data3 -133.3333 0.9271 -1229.8967  963.2301  False
 data2  data3    -320.0 0.6626 -1416.5634  776.5634  False
----------------------------------------------------------


4. Skoro p-value ANOVA = 0.684 (nieistotne), test Tukey'a potwierdził, że żadna para grup się nie różni

## Zadanie

In [42]:
import pandas as pd

# 1. Wczytaj dane zawierające wyniki wyborów prezydenckich w Rosji z 2020r.
df= pd.read_csv("dataset/russia2020_vote.csv")

# 2. Wyznacz frekwencję w poszczególnych okręgach oraz procentową liczbę głosów za:
df['frekwencja'] = df['given']/np.maximum(1, df['nominal'])
df['glosow_za'] = df['yes']/np.maximum(1, df['given'])

# 3. Stosując analizę ANOVA wskaż regiony dla których wynik wyborczy nie różni się istotnie od wyniku uzyskanego w Moskwie
regions = df['region_en'].unique()
region_groups = [df[df['region_en'] == region]['glosow_za'].values for region in regions]

f_value, p_value = f_oneway(*region_groups)
print(f'One-way ANOVA dla wszystkich regionów:')
print(f'F-statistic: {f_value:.4f}, p-value: {p_value:.6f}')




One-way ANOVA dla wszystkich regionów:
F-statistic: 731.5662, p-value: 0.000000


Na podstawie testu one-hot Anova widać, że istnieją istotne statystycznie różnice między wynikami wyborczymi w różnych regionach Rosji

In [43]:
tukey_result = pairwise_tukeyhsd(
    endog=df['glosow_za'],
    groups=df['region_en'],
    alpha=0.05
)

# Porównania z Moskovskaya
tukey_df = pd.DataFrame(data=tukey_result.summary().data[1:], 
                        columns=tukey_result.summary().data[0])
moscow_comp = tukey_df[(tukey_df['group1'] == 'Moskovskaya') | 
                        (tukey_df['group2'] == 'Moskovskaya')]

# Regiony BEZ istotnej różnicy (reject = False)
no_diff = moscow_comp[moscow_comp['reject'] == False]
print(f'\nRegiony bez istotnej różnicy od Moskovskaya ({len(no_diff)}):\n')
print(no_diff[['group1', 'group2', 'meandiff', 'p-adj']])


Regiony bez istotnej różnicy od Moskovskaya (7):

                          group1         group2  meandiff   p-adj
997                   Chukotskiy    Moskovskaya   -0.0329  0.9981
1213                  Evreyskaya    Moskovskaya   -0.0148  1.0000
1283  Gorod Baykonur (Kazahstan)    Moskovskaya    0.1547  0.0796
2570                    Kurskaya    Moskovskaya    0.0071  1.0000
2617              Leningradskaya    Moskovskaya   -0.0106  0.8632
2752                    Mariy El    Moskovskaya    0.0098  1.0000
2879                 Moskovskaya  Zabaykal'skiy   -0.0038  1.0000


4. Przeanalizuj wyniki częstość głosów za, analizując częstotliwość występowania cyfr na drugim miejscu po przecinku

In [46]:
df['druga_cyfra'] = (df['glosow_za'] * 100).astype(int) % 10 

obs = df['druga_cyfra'].value_counts().sort_index()
# Upewnij się, że masz wszystkie cyfry (nawet jeśli wystąpiły 0 razy)
obs = obs.reindex(range(10), fill_value=0)
n = len(df)
exp = [n / 10] * 10
chi2, p = chisquare(obs, exp) 
print(f"Chi-squared: {chi2}, p-value: {p}")


Chi-squared: 313.37203534335765, p-value: 3.791717068991187e-62


Chi-squared: 313.37203534335765, p-value: 3.791717068991187e-62
Odrzucamy hipotezę o jednostajności. Oznacza to, że niektóre cyfry występują nienaturalnie często